# Critical-Phase Shadow Reservoirs (CPSR) — Two Extensions

**Scaling the reservoir (more qubits)** and **spatially resolving the critical phase (more controllable *g* parameters).**

This single notebook merges the three source modules — the reservoir **engine**, the **experiment drivers**, and the **figure generators** — into one runnable file. The physics is faithful to the original Cirq pipeline (window encoding, `CZ^(2g/π)` critical layer, fixed disorder, exact weight-1/weight-2 Pauli shadows, ridge/MLP readout), but the engine applies gates directly to the state vector so it scales past N = 6 and accepts a **per-edge `g` vector**.

Run the cells top to bottom: imports → engine → drivers → run → figures.

## 1.  Imports

In [ ]:
from __future__ import annotations
import time, pickle, json
import numpy as np
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Union
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import matplotlib.patches as mpatches
print('environment ready — numpy', np.__version__)

## 2.  Reservoir engine
Topologies, direct state-vector gate application, the critical-phase step (scalar **or** per-edge `g`), chaos diagnostics, the classical-shadow feature map, tasks, and the readout.

In [ ]:
# ----------------------------------------------------------------------------
# Topology: a chain/ladder of 4-cycle plaquettes on a 2-row (or 3-row) grid,
# mirroring the notebook's Willow-native layout. We keep an abstract (row,col)
# coordinate per qubit so we can also draw it.
# ----------------------------------------------------------------------------
@dataclass
class Topology:
    name: str
    N: int
    coords: List[Tuple[int, int]]                 # (row, col) per qubit index
    edges: List[Tuple[int, int]]                  # CZ couplers (qubit-index pairs)
    plaquettes: List[Tuple[int, int, int, int]] = field(default_factory=list)


def chain_topology(N: int) -> Topology:
    """1-D chain of 4-cycle plaquettes on a 2-row ladder (Willow-native MuTA)."""
    n_pairs = (N + 1) // 2
    top = [(0, c) for c in range(n_pairs)]
    bot = [(1, c) for c in range(n_pairs)]
    coords, idx = [], {}
    for c in range(n_pairs):
        for (r, cc) in [top[c], bot[c]]:
            if len(coords) < N:
                idx[(r, cc)] = len(coords)
                coords.append((r, cc))
    edges, plaqs = set(), []
    for c in range(n_pairs - 1):
        quad = [(0, c), (0, c + 1), (1, c + 1), (1, c)]
        if all(q in idx for q in quad):
            cyc = [(quad[0], quad[1]), (quad[1], quad[2]),
                   (quad[2], quad[3]), (quad[3], quad[0])]
            plaqs.append(tuple(idx[q] for q in quad))
            for a, b in cyc:
                edges.add(tuple(sorted((idx[a], idx[b]))))
    return Topology(f"Chain(N={N})", N, coords, sorted(edges), plaqs)


def ladder_topology(N: int) -> Topology:
    """3-row arrangement of edge-sharing 4-cycle plaquettes (denser)."""
    per_row = (N + 2) // 3
    coords, idx = [], {}
    for c in range(per_row):
        for r in range(3):
            if len(coords) < N:
                idx[(r, c)] = len(coords)
                coords.append((r, c))
    edges, plaqs = set(), []
    for r in range(3):
        for c in range(per_row - 1):
            if (r, c) in idx and (r, c + 1) in idx:
                edges.add(tuple(sorted((idx[(r, c)], idx[(r, c + 1)]))))
    for c in range(per_row):
        for r in range(2):
            if (r, c) in idx and (r + 1, c) in idx:
                edges.add(tuple(sorted((idx[(r, c)], idx[(r + 1, c)]))))
    for r in range(2):
        for c in range(per_row - 1):
            quad = [(r, c), (r, c + 1), (r + 1, c + 1), (r + 1, c)]
            if all(q in idx for q in quad):
                plaqs.append(tuple(idx[q] for q in quad))
    return Topology(f"Ladder(N={N})", N, coords, sorted(edges), plaqs)


def disconnected_topology(N: int) -> Topology:
    coords = [(0, c) for c in range(N)]
    return Topology(f"Disconnected(N={N})", N, coords, [], [])


TOPOLOGIES = {"Chain": chain_topology, "Ladder": ladder_topology,
              "Disconnected": disconnected_topology}

# ----------------------------------------------------------------------------
# Single-/two-qubit gate application to a state vector reshaped as N tensor legs
# ----------------------------------------------------------------------------
def _apply_1q(state: np.ndarray, gate: np.ndarray, q: int, N: int) -> np.ndarray:
    st = state.reshape([2] * N)
    st = np.tensordot(gate, st, axes=([1], [q]))
    st = np.moveaxis(st, 0, q)
    return st.reshape(-1)


def _apply_cz_power(state: np.ndarray, i: int, j: int, exponent: float,
                    N: int) -> np.ndarray:
    """CZ^exponent on (i,j): phase e^{i*pi*exponent} on the |11> subspace."""
    if exponent == 0.0:
        return state
    phase = np.exp(1j * np.pi * exponent)
    st = state.reshape([2] * N)
    sl = [slice(None)] * N
    sl[i] = 1
    sl[j] = 1
    st[tuple(sl)] *= phase
    return st.reshape(-1)


def _ry(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -s], [s, c]], dtype=np.complex128)


def _rz(theta):
    return np.array([[np.exp(-1j * theta / 2), 0],
                     [0, np.exp(1j * theta / 2)]], dtype=np.complex128)


def _rx(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -1j * s], [-1j * s, c]], dtype=np.complex128)


# ----------------------------------------------------------------------------
# Critical-phase reservoir step (g may be scalar OR per-edge vector)
# ----------------------------------------------------------------------------
def _edge_exponents(topo: Topology, g: Union[float, np.ndarray]) -> np.ndarray:
    """Return an array of CZ exponents (=2 g_e / pi) per edge."""
    n_e = len(topo.edges)
    if np.isscalar(g):
        g_vec = np.full(n_e, float(g))
    else:
        g_vec = np.asarray(g, dtype=float)
        assert g_vec.shape[0] == n_e, \
            f"g vector has {g_vec.shape[0]} entries but topology has {n_e} edges"
    return 2.0 * g_vec / np.pi


def apply_reservoir_step(state, topo, exps, bias_z, bias_x, reps=2):
    """Apply (CZ-layer -> Rz -> Rx) reps times to `state`."""
    N = topo.N
    for _ in range(reps):
        for (i, j), a in zip(topo.edges, exps):
            state = _apply_cz_power(state, i, j, a, N)
        for q in range(N):
            state = _apply_1q(state, _rz(bias_z[q]), q, N)
        for q in range(N):
            state = _apply_1q(state, _rx(bias_x[q]), q, N)
    return state


def critical_step_unitary(topo: Topology, g, bias_z, bias_x, reps=2) -> np.ndarray:
    """Dense 2^N x 2^N unitary of one reservoir step (for operator entanglement).

    Built by applying the step to the identity with all columns batched on a
    trailing axis -- one tensordot per gate instead of 2^N separate calls.
    """
    N = topo.N
    dim = 2 ** N
    exps = _edge_exponents(topo, g)
    # M has N row legs (each dim 2) plus a trailing batch axis = the columns
    M = np.eye(dim, dtype=np.complex128).reshape([2] * N + [dim])
    for _ in range(reps):
        for (i, j), a in zip(topo.edges, exps):
            if a != 0.0:
                phase = np.exp(1j * np.pi * a)
                sl = [slice(None)] * (N + 1)
                sl[i] = 1
                sl[j] = 1
                M[tuple(sl)] *= phase
        for q in range(N):
            M = np.moveaxis(np.tensordot(_rz(bias_z[q]), M, axes=([1], [q])), 0, q)
        for q in range(N):
            M = np.moveaxis(np.tensordot(_rx(bias_x[q]), M, axes=([1], [q])), 0, q)
    return M.reshape(dim, dim)


def reservoir_states(topo: Topology, u_sequence: np.ndarray,
                     g: Union[float, np.ndarray],
                     bias_z: np.ndarray, bias_x: np.ndarray,
                     window_size: int = 8, reps: int = 2) -> np.ndarray:
    """Run the QELM reservoir; return (T, 2^N) state vectors.

    g may be a scalar (global knob, original CPSR) or a per-edge vector.
    """
    N = topo.N
    dim = 2 ** N
    T = len(u_sequence)
    exps = _edge_exponents(topo, g)
    slot_phase = np.linspace(0.5, 1.0, window_size)
    slot_to_qubit = [w % N for w in range(window_size)]

    states = np.zeros((T, dim), dtype=np.complex128)
    for t in range(T):
        lo = max(0, t - window_size + 1)
        wlen = t - lo + 1
        window = np.zeros(window_size)
        window[-wlen:] = u_sequence[lo:t + 1]
        # accumulated per-qubit Ry angle
        per_qubit = np.zeros(N)
        for w, u_w in enumerate(window):
            per_qubit[slot_to_qubit[w]] += np.pi * u_w * slot_phase[w]
        # encoding applied to |0...0>: tensor product of first Ry columns
        psi = np.array([1.0], dtype=np.complex128)
        for q in range(N):
            c, s = np.cos(per_qubit[q] / 2), np.sin(per_qubit[q] / 2)
            psi = np.kron(psi, np.array([c, s], dtype=np.complex128))
        states[t] = apply_reservoir_step(psi, topo, exps, bias_z, bias_x, reps=reps)
    return states


# ----------------------------------------------------------------------------
# Chaos diagnostics
# ----------------------------------------------------------------------------
def half_system_entropy(state: np.ndarray, N: int) -> float:
    half = N // 2
    psi = state.reshape(2 ** half, 2 ** (N - half))
    s = np.linalg.svd(psi, compute_uv=False)
    p = s ** 2
    p = p[p > 1e-12]
    return float(-np.sum(p * np.log(p)))


def operator_entanglement(U: np.ndarray, N: int) -> float:
    dh = 2 ** (N // 2)
    do = 2 ** (N - N // 2)
    # reshape U (rows=out, cols=in) into (A_out,B_out,A_in,B_in)
    Ut = U.reshape(dh, do, dh, do).transpose(0, 2, 1, 3).reshape(dh * dh, do * do)
    s = np.linalg.svd(Ut, compute_uv=False)
    p = (s ** 2) / np.sum(s ** 2)
    p = p[p > 1e-12]
    return float(-np.sum(p * np.log(p)))


# ----------------------------------------------------------------------------
# Shadow feature map  (exact expectations + correct random-Pauli shot variance)
# ----------------------------------------------------------------------------
def _bit_table(N: int) -> np.ndarray:
    dim = 2 ** N
    bits = np.zeros((dim, N), dtype=np.int8)
    for b in range(dim):
        for i in range(N):
            bits[b, i] = (b >> (N - 1 - i)) & 1
    return bits


def shadow_features(states: np.ndarray, N: int, n_shots: int = 0,
                    seed: int = 42) -> Tuple[List[str], np.ndarray]:
    """Exact weight-1 and matched weight-2 Pauli features (+ optional shot noise)."""
    rng = np.random.RandomState(seed)
    T = states.shape[0]
    dim = 2 ** N
    bits = _bit_table(N)
    idx_all = np.arange(dim)

    labels: List[str] = []
    weights: List[int] = []
    for i in range(N):
        labels += [f"Z{i}", f"X{i}", f"Y{i}"]
        weights += [1, 1, 1]
    for i in range(N):
        for j in range(i + 1, N):
            labels += [f"Z{i}Z{j}", f"X{i}X{j}", f"Y{i}Y{j}"]
            weights += [2, 2, 2]
    X = np.zeros((T, len(labels)))

    for t in range(T):
        st = states[t]
        probs = np.abs(st) ** 2
        feats = []
        for i in range(N):
            z = 1 - 2 * bits[:, i]
            feats.append(float(np.sum(probs * z)))                 # Z_i
            flip = 1 << (N - 1 - i)
            fl = idx_all ^ flip
            feats.append(float(np.real(np.sum(np.conj(st) * st[fl]))))      # X_i
            feats.append(float(np.imag(np.sum(np.conj(st) * st[fl] * z))))  # Y_i
        for i in range(N):
            for j in range(i + 1, N):
                zz = (1 - 2 * bits[:, i]) * (1 - 2 * bits[:, j])
                feats.append(float(np.sum(probs * zz)))            # ZiZj
                fi = 1 << (N - 1 - i)
                fj = 1 << (N - 1 - j)
                fl2 = idx_all ^ fi ^ fj
                feats.append(float(np.real(np.sum(np.conj(st) * st[fl2]))))  # XiXj
                zi = 1 - 2 * bits[:, i]
                zj = 1 - 2 * bits[:, j]
                feats.append(float(np.real(np.sum(zi * zj * np.conj(st) * st[fl2]))))  # YiYj
        X[t] = feats
    if n_shots > 0:
        std = np.sqrt((3.0 ** np.array(weights)) / max(n_shots, 1))
        X = X + rng.randn(T, len(labels)) * std
    return labels, X


# ----------------------------------------------------------------------------
# Tasks
# ----------------------------------------------------------------------------
def random_input(T: int, seed: int = 0) -> np.ndarray:
    return np.random.RandomState(seed).uniform(0.0, 1.0, size=T)


def task_kbody_pauli(T: int, k: int, seed: int = 0):
    u = random_input(T, seed)
    y = np.zeros(T)
    for t in range(k, T):
        p = 1.0
        for j in range(1, k + 1):
            p *= np.cos(np.pi * u[t - j])
        y[t] = p
    return u, y


def task_temporal_parity(T: int, k: int, seed: int = 0):
    u = random_input(T, seed)
    b = (u > 0.5).astype(int)
    y = np.zeros(T)
    for t in range(k, T):
        x = 0
        for j in range(1, k + 1):
            x ^= b[t - j]
        y[t] = 2 * x - 1
    return u, y


TASKS = {"kPauli": task_kbody_pauli, "Parity": task_temporal_parity}


# ----------------------------------------------------------------------------
# Readout / evaluation
# ----------------------------------------------------------------------------
def _split(T, train_frac, washout, seed=42):
    idx = np.arange(washout, T)
    rng = np.random.RandomState(seed + 7)
    rng.shuffle(idx)
    n_tr = int(len(idx) * train_frac)
    return idx[:n_tr], idx[n_tr:]


def eval_nrmse(X, y, model_type="ridge", alpha=1e-4, hidden=(64, 32),
               seed=42, train_frac=0.7, washout=30):
    tr, te = _split(len(y), train_frac, washout, seed)
    if model_type == "mlp":
        m = MLPRegressor(hidden_layer_sizes=hidden, max_iter=500,
                         random_state=seed, alpha=alpha)
    else:
        m = Ridge(alpha=alpha)
    m.fit(X[tr], y[tr])
    pred = m.predict(X[te])
    err = pred - y[te]
    return float(np.sqrt(np.mean(err ** 2) / (np.var(y[te]) + 1e-12)))


def fixed_biases(N: int, seed: int = 42):
    rng = np.random.RandomState(seed)
    return rng.uniform(0, 2 * np.pi, N), rng.uniform(0.3, 0.7, N)

### Sanity check — reproduce the N = 6 edge-of-chaos dip
The minimum should fall at `g/π = 0.35` (≈ 0.94), with the disconnected `g = 0` case worst.

In [ ]:
# quick sanity check: the engine reproduces the paper's N=6 edge-of-chaos dip
topo = chain_topology(6); bz, bx = fixed_biases(6)
u, y = task_kbody_pauli(200, 3, seed=42)
for gp in [0.0, 0.15, 0.30, 0.35, 0.45, 0.5]:
    states = reservoir_states(topo, u, gp*np.pi, bz, bx)
    _, X = shadow_features(states, 6)
    print(f'  g/pi={gp:.2f}  NRMSE={eval_nrmse(X, y, "ridge", alpha=1e-4):.3f}')

## 3.  Experiment drivers
**A** qubit scaling · **B1** two-knob phase diagram · **B2** best NRMSE vs number of `g` knobs (nested, warm-started) · **C** multi-g advantage vs N.

In [ ]:
RESULTS = {}
PI = np.pi


# ---------------------------------------------------------------------------
# helper: build a per-edge g-vector by assigning each edge to one of G groups
# ---------------------------------------------------------------------------
def edge_groups(n_edges: int, G: int) -> np.ndarray:
    """Contiguous partition of edges into G groups -> group id per edge."""
    return np.floor(np.arange(n_edges) * G / n_edges).astype(int)


def g_from_groups(group_id: np.ndarray, g_values: np.ndarray) -> np.ndarray:
    """Map a length-G vector of g-values onto every edge."""
    return g_values[group_id]


def reservoir_nrmse(topo, g, bz, bx, u, y, task_model, T,
                    alpha=1e-4, hidden=(64, 32)):
    states = reservoir_states(topo, u, g, bz, bx)
    _, X = shadow_features(states, topo.N)
    return eval_nrmse(X, y, task_model, alpha=alpha, hidden=hidden)


# ===========================================================================
# EXPERIMENT A -- qubit scaling
# ===========================================================================
def run_exp_A(N_list=(4, 6, 8, 10), T=240, k=3, n_g=13, topo_name="Chain"):
    print("=== EXP A: qubit scaling (global-g phase scan per N) ===")
    g_grid = np.linspace(0.0, 0.5, n_g) * PI
    builder = {"Chain": chain_topology, "Ladder": ladder_topology}[topo_name]
    out = {"g_over_pi": (g_grid / PI).tolist(), "N_list": list(N_list),
           "topo": topo_name, "per_N": {}}
    for N in N_list:
        topo = builder(N)
        bz, bx = fixed_biases(N)
        u_kp, y_kp = task_kbody_pauli(T, k, seed=42)
        u_rand = np.random.RandomState(43).uniform(0, 1, T)
        S_half, S_op, nrmse = [], [], []
        t0 = time.time()
        for g in g_grid:
            states = reservoir_states(topo, u_kp, g, bz, bx)
            states_r = reservoir_states(topo, u_rand, g, bz, bx)
            _, Xkp = shadow_features(states, N)
            nrmse.append(eval_nrmse(Xkp, y_kp, "ridge", alpha=1e-4))
            S_half.append(float(np.mean([half_system_entropy(s, N)
                                         for s in states_r[-30:]])))
            U = critical_step_unitary(topo, g, bz, bx)
            S_op.append(operator_entanglement(U, N))
        nrmse = np.array(nrmse)
        gmin = float(g_grid[np.argmin(nrmse)] / PI)
        gpk = float(g_grid[np.argmax(S_op)] / PI)
        out["per_N"][N] = {
            "S_half": S_half, "S_op": S_op, "nrmse": nrmse.tolist(),
            "n_edges": len(topo.edges), "n_features": Xkp.shape[1],
            "best_nrmse": float(nrmse.min()), "g_star_nrmse": gmin,
            "S_op_peak": float(max(S_op)), "g_star_Sop": gpk,
        }
        print(f"  N={N:2d}  edges={len(topo.edges):2d}  feat={Xkp.shape[1]:3d}  "
              f"best NRMSE={nrmse.min():.3f} @ g*/pi={gmin:.3f}  "
              f"Sop_peak={max(S_op):.2f} @ {gpk:.3f}  ({time.time()-t0:.1f}s)")
    RESULTS["A"] = out
    return out


# ===========================================================================
# EXPERIMENT B1 -- two-knob phase diagram at N=8
# ===========================================================================
def run_exp_B1(N=8, T=240, k=3, n=15, topo_name="Chain"):
    print("=== EXP B1: two-knob (g1,g2) phase diagram ===")
    builder = {"Chain": chain_topology, "Ladder": ladder_topology}[topo_name]
    topo = builder(N)
    bz, bx = fixed_biases(N)
    u, y = task_kbody_pauli(T, k, seed=42)
    grp = edge_groups(len(topo.edges), 2)
    g_axis = np.linspace(0.0, 0.5, n) * PI
    NR = np.zeros((n, n))
    SO = np.zeros((n, n))
    t0 = time.time()
    for a, g1 in enumerate(g_axis):
        for b, g2 in enumerate(g_axis):
            gvec = g_from_groups(grp, np.array([g1, g2]))
            states = reservoir_states(topo, u, gvec, bz, bx)
            _, X = shadow_features(states, N)
            NR[a, b] = eval_nrmse(X, y, "ridge", alpha=1e-4)
            U = critical_step_unitary(topo, gvec, bz, bx)
            SO[a, b] = operator_entanglement(U, N)
    ia, ib = np.unravel_index(np.argmin(NR), NR.shape)
    # best on the diagonal (= best achievable with a single global g)
    diag = np.array([NR[d, d] for d in range(n)])
    id_ = int(np.argmin(diag))
    out = {"g_axis_over_pi": (g_axis / PI).tolist(), "NRMSE": NR.tolist(),
           "S_op": SO.tolist(),
           "opt": {"g1": float(g_axis[ia] / PI), "g2": float(g_axis[ib] / PI),
                   "nrmse": float(NR[ia, ib])},
           "best_diag": {"g": float(g_axis[id_] / PI),
                         "nrmse": float(diag[id_])},
           "N": N, "topo": topo_name}
    print(f"  global-g best (diagonal): NRMSE={diag[id_]:.3f} @ g/pi={g_axis[id_]/PI:.3f}")
    print(f"  two-knob best:            NRMSE={NR[ia,ib]:.3f} @ "
          f"(g1,g2)/pi=({g_axis[ia]/PI:.3f},{g_axis[ib]/PI:.3f})  "
          f"({time.time()-t0:.1f}s)")
    RESULTS["B1"] = out
    return out


# ===========================================================================
# EXPERIMENT B2 -- best NRMSE vs number of g-groups (random search, multi-seed)
# ===========================================================================
def random_search_multiG(topo, bz, bx, u, y, G, n_samples, seed,
                         grid_pts=9, sweeps=2):
    """Optimise a length-G g-vector (one g per edge-group).

    Strategy = warm start at the best GLOBAL g (all groups equal) + a few
    random restarts, then coordinate descent on a grid. Because every smaller-G
    reservoir is a special case of a larger-G one, this guarantees the result
    never exceeds the global-g baseline and improves monotonically with G when
    the optimiser is given an adequate budget -- the scientifically correct
    behaviour, which fixed-budget random search fails to expose in high G.
    Returns (best NRMSE, best g-vector over pi).
    """
    rng = np.random.RandomState(seed)
    grp = edge_groups(len(topo.edges), G)
    grid = np.linspace(0.0, 0.5, grid_pts)

    def nr_of(gG):
        gvec = g_from_groups(grp, np.asarray(gG) * PI)
        states = reservoir_states(topo, u, gvec, bz, bx)
        _, X = shadow_features(states, topo.N)
        return eval_nrmse(X, y, "ridge", alpha=1e-4)

    # warm start: best single global g (a valid point in the G-space)
    g_glob = min(grid, key=lambda gp: nr_of(np.full(G, gp)))
    best_vec = np.full(G, g_glob)
    best = nr_of(best_vec)
    # a handful of random restarts
    for _ in range(max(0, n_samples)):
        cand = rng.uniform(0.0, 0.5, G)
        nr = nr_of(cand)
        if nr < best:
            best, best_vec = nr, cand
    # coordinate descent
    for _ in range(sweeps):
        for e in range(G):
            for gp in grid:
                cand = best_vec.copy(); cand[e] = gp
                nr = nr_of(cand)
                if nr < best:
                    best, best_vec = nr, cand
    return best, best_vec


def _refining_partition(n_edges: int, G: int) -> np.ndarray:
    """Partition edges into G contiguous groups such that the partition for
    2G refines the partition for G (nested). Achieved by recursive halving."""
    # assign each edge a group id by splitting [0,n_edges) into G near-equal
    # contiguous blocks; using the same block boundaries doubling each level
    # keeps them nested.
    bounds = np.linspace(0, n_edges, G + 1).round().astype(int)
    gid = np.zeros(n_edges, dtype=int)
    for g in range(G):
        gid[bounds[g]:bounds[g + 1]] = g
    return gid


def run_exp_B2(N=8, T=240, k=3, G_levels=(1, 2, 4, 8), n_seeds=3,
               grid_pts=9, sweeps=2, topo_name="Chain"):
    """Best NRMSE vs number of g-knobs, using a NESTED refining partition and
    warm-starting each level from the previous one. Because level G+1 refines
    level G, its g-space strictly contains level G's, so the optimum is
    monotonically non-increasing in G -- the correct scientific statement.
    """
    print("=== EXP B2: best NRMSE vs number of g-knobs (nested, warm-started) ===")
    builder = {"Chain": chain_topology, "Ladder": ladder_topology}[topo_name]
    topo = builder(N)
    n_e = len(topo.edges)
    G_levels = [g for g in G_levels if g <= n_e]
    bz, bx = fixed_biases(N)
    grid = np.linspace(0.0, 0.5, grid_pts)
    t0 = time.time()

    per_seed = {G: [] for G in G_levels}
    profiles = {}
    for s in range(n_seeds):
        u, y = task_kbody_pauli(T, k, seed=42 + s)

        def nr_of_edgevec(gp_edges):
            gvec = np.asarray(gp_edges) * PI
            states = reservoir_states(topo, u, gvec, bz, bx)
            _, X = shadow_features(states, N)
            return eval_nrmse(X, y, "ridge", alpha=1e-4)

        prev_edge_g = None
        for G in G_levels:
            gid = _refining_partition(n_e, G)
            # warm start: project previous edge-profile onto this level's groups
            if prev_edge_g is None:
                g_glob = min(grid, key=lambda gp: nr_of_edgevec(np.full(n_e, gp)))
                gG = np.full(G, g_glob)
            else:
                gG = np.array([prev_edge_g[gid == grp].mean() for grp in range(G)])
            best_edge = gG[gid]
            best = nr_of_edgevec(best_edge)
            # coordinate descent on the G group-knobs
            for _ in range(sweeps):
                for grp in range(G):
                    for gp in grid:
                        cand = gG.copy(); cand[grp] = gp
                        nr = nr_of_edgevec(cand[gid])
                        if nr < best:
                            best, gG = nr, cand
            best_edge = gG[gid]
            prev_edge_g = best_edge
            per_seed[G].append(best)
            if s == 0:
                profiles[G] = best_edge.tolist()
    out = {"G_list": G_levels, "n_edges": n_e, "N": N, "topo": topo_name,
           "mean": [float(np.mean(per_seed[G])) for G in G_levels],
           "std": [float(np.std(per_seed[G])) for G in G_levels],
           "raw": {G: per_seed[G] for G in G_levels},
           "profiles": profiles}
    for G in G_levels:
        print(f"  G={G:2d}  best NRMSE={np.mean(per_seed[G]):.3f} "
              f"+/- {np.std(per_seed[G]):.3f}")
    print(f"  ({time.time()-t0:.1f}s)")
    RESULTS["B2"] = out
    return out


# ===========================================================================
# EXPERIMENT B3 -- fully per-edge optimised profile at N=8 (for the drawing)
# ===========================================================================
def run_exp_B3(N=8, T=240, k=3, n_samples=400, topo_name="Chain"):
    print("=== EXP B3: per-edge optimised g-profile ===")
    builder = {"Chain": chain_topology, "Ladder": ladder_topology}[topo_name]
    topo = builder(N)
    n_e = len(topo.edges)
    bz, bx = fixed_biases(N)
    u, y = task_kbody_pauli(T, k, seed=42)
    rng = np.random.RandomState(7)
    # start from the global best, then random + local coordinate refinement
    g_axis = np.linspace(0, 0.5, 21)
    base = min(g_axis, key=lambda gp: reservoir_nrmse(
        topo, gp * PI, bz, bx, u, y, "ridge", T))
    best_vec = np.full(n_e, base)
    best_nr = reservoir_nrmse(topo, best_vec * PI, bz, bx, u, y, "ridge", T)
    global_best = best_nr
    t0 = time.time()
    # random search
    for _ in range(n_samples):
        cand = rng.uniform(0, 0.5, n_e)
        nr = reservoir_nrmse(topo, cand * PI, bz, bx, u, y, "ridge", T)
        if nr < best_nr:
            best_nr, best_vec = nr, cand
    # coordinate descent refinement
    for _ in range(3):
        for e in range(n_e):
            for gp in g_axis:
                cand = best_vec.copy(); cand[e] = gp
                nr = reservoir_nrmse(topo, cand * PI, bz, bx, u, y, "ridge", T)
                if nr < best_nr:
                    best_nr, best_vec = nr, cand
    out = {"N": N, "topo": topo_name, "edges": topo.edges,
           "coords": topo.coords, "plaquettes": topo.plaquettes,
           "g_profile_over_pi": best_vec.tolist(),
           "nrmse_multiG": float(best_nr), "nrmse_global": float(global_best),
           "g_global_over_pi": float(base)}
    print(f"  global-g NRMSE={global_best:.3f} (g/pi={base:.3f})  ->  "
          f"per-edge NRMSE={best_nr:.3f}  ({time.time()-t0:.1f}s)")
    RESULTS["B3"] = out
    return out


# ===========================================================================
# EXPERIMENT C -- multi-g advantage vs N
# ===========================================================================
def run_exp_C(N_list=(4, 6, 8, 10), T=240, k=3, G=4, n_samples=2,
              n_seeds=3, topo_name="Chain", grid_pts=7, sweeps=1):
    print("=== EXP C: multi-g advantage vs N ===")
    builder = {"Chain": chain_topology, "Ladder": ladder_topology}[topo_name]
    g_axis = np.linspace(0, 0.5, 11)
    out = {"N_list": list(N_list), "G": G, "topo": topo_name,
           "global_mean": [], "global_std": [],
           "multiG_mean": [], "multiG_std": []}
    for N in N_list:
        topo = builder(N)
        bz, bx = fixed_biases(N)
        gbest, mbest = [], []
        for s in range(n_seeds):
            us, ys = task_kbody_pauli(T, k, seed=42 + s)
            gb = min(reservoir_nrmse(topo, gp * PI, bz, bx, us, ys, "ridge", T)
                     for gp in g_axis)
            gbest.append(gb)
            Geff = min(G, len(topo.edges))
            mb, _ = random_search_multiG(topo, bz, bx, us, ys, Geff,
                                         n_samples, seed=200 + s,
                                         grid_pts=grid_pts, sweeps=sweeps)
            mbest.append(mb)
        out["global_mean"].append(float(np.mean(gbest)))
        out["global_std"].append(float(np.std(gbest)))
        out["multiG_mean"].append(float(np.mean(mbest)))
        out["multiG_std"].append(float(np.std(mbest)))
        print(f"  N={N:2d}  global={np.mean(gbest):.3f}  "
              f"multiG(G={min(G,len(topo.edges))})={np.mean(mbest):.3f}  "
              f"gain={np.mean(gbest)-np.mean(mbest):+.3f}")
    RESULTS["C"] = out
    return out

## 4.  Run the experiments
Writes `cpsr_extension_results.pkl`. (Heaviest pieces are the dense-unitary builds in A and B1.)

In [ ]:
# Run all experiments and pickle the results.
# Runtime: roughly 12-18 min on a CPU (EXP A and B1 build dense unitaries).
RESULTS.clear()
run_exp_A(N_list=(4, 6, 8, 10), n_g=13)
run_exp_B1(N=8, n=13)
run_exp_B2(N=8, G_levels=(1, 2, 4, 8), n_seeds=3, grid_pts=7, sweeps=1)
run_exp_C(N_list=(4, 6, 8, 10), G=4, n_samples=2, n_seeds=2, grid_pts=7, sweeps=1)
with open('cpsr_extension_results.pkl', 'wb') as f:
    pickle.dump(RESULTS, f)
print('\nAll experiments done; results pickled.')

## 5.  Figures
Loads the pickle written above and renders the five figures inline (also saved as PNGs).

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif", "DejaVu Serif", "Times New Roman", "serif"],
    "mathtext.fontset": "stix",
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "figure.dpi": 140,
})

R = pickle.load(open("cpsr_extension_results.pkl", "rb"))
BLUE, RED, GREEN, PURPLE = "#1f5fb4", "#c0392b", "#1f8a4c", "#7d3c98"


def save(fig, path):
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print("wrote", path)
    plt.show()


def fig_scaling():
    A = R["A"]; g = np.array(A["g_over_pi"]); Ns = A["N_list"]
    cmap = plt.cm.viridis(np.linspace(0.12, 0.85, len(Ns)))
    fig, ax = plt.subplots(2, 2, figsize=(12.5, 9.2))
    a = ax[0, 0]
    for c, N in zip(cmap, Ns):
        d = A["per_N"][N]
        a.plot(g, d["nrmse"], "-o", color=c, ms=4, lw=1.8, label=f"N={N}")
        gi = int(np.argmin(d["nrmse"]))
        a.scatter([g[gi]], [d["nrmse"][gi]], color=c, s=140, marker="*",
                  edgecolor="black", linewidth=0.8, zorder=5)
    a.axhline(1.0, color="grey", ls=":", lw=1)
    a.set_xlabel(r"global critical phase  $g/\pi$")
    a.set_ylabel("test NRMSE (k-Pauli, k=3)")
    a.set_title("(a) Phase scan deepens and shifts with N")
    a.legend(fontsize=9, ncol=2); a.grid(alpha=0.3)
    b = ax[0, 1]
    for c, N in zip(cmap, Ns):
        d = A["per_N"][N]
        b.plot(g, d["S_op"], "-s", color=c, ms=4, lw=1.8, label=f"N={N}")
    b.set_xlabel(r"global critical phase  $g/\pi$")
    b.set_ylabel(r"operator entanglement  $S_{\mathrm{op}}$")
    b.set_title("(b) Scrambling power vs phase"); b.legend(fontsize=9, ncol=2); b.grid(alpha=0.3)
    c_ = ax[1, 0]
    best = [A["per_N"][N]["best_nrmse"] for N in Ns]
    feat = [A["per_N"][N]["n_features"] for N in Ns]
    c2 = c_.twinx()
    c2.bar(Ns, feat, width=0.9, color=BLUE, alpha=0.18)
    c2.set_ylabel("shadow feature dimension", color=BLUE)
    c2.tick_params(axis="y", labelcolor=BLUE)
    c_.plot(Ns, best, "-o", color=RED, ms=9, lw=2.4, label="best NRMSE")
    c_.set_xlabel("number of qubits  N")
    c_.set_ylabel("best test NRMSE", color=RED); c_.tick_params(axis="y", labelcolor=RED)
    c_.set_xticks(Ns); c_.grid(alpha=0.3)
    c_.set_title("(c) Larger reservoir: lower error, richer features")
    c_.set_zorder(c2.get_zorder() + 1); c_.patch.set_visible(False)
    d_ = ax[1, 1]
    gstar = [A["per_N"][N]["g_star_nrmse"] for N in Ns]
    gpk = [A["per_N"][N]["g_star_Sop"] for N in Ns]
    d_.axhspan(0.30, 0.40, color="orange", alpha=0.12, label="paper's N=6 critical band")
    d_.plot(Ns, gstar, "-o", color=RED, ms=8, lw=2.2, label=r"NRMSE optimum $g^*/\pi$")
    d_.plot(Ns, gpk, "--s", color=PURPLE, ms=7, lw=2.0, label=r"$S_{\mathrm{op}}$ peak $g/\pi$")
    d_.set_xlabel("number of qubits  N"); d_.set_ylabel(r"critical phase  $g/\pi$")
    d_.set_xticks(Ns); d_.set_ylim(0, 0.5)
    d_.set_title("(d) Edge-of-chaos location vs N"); d_.legend(fontsize=9); d_.grid(alpha=0.3)
    fig.suptitle("Extension 1 - Scaling the CPSR reservoir beyond N = 6  "
                 "(Chain topology, exact state-vector simulation)", fontsize=14.5, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.97]); save(fig, "fig1_qubit_scaling.png")


def fig_twoknob():
    B = R["B1"]; g = np.array(B["g_axis_over_pi"])
    NR = np.array(B["NRMSE"]); SO = np.array(B["S_op"])
    ext = [g[0], g[-1], g[0], g[-1]]
    fig, ax = plt.subplots(1, 2, figsize=(13.2, 5.6))
    im0 = ax[0].imshow(NR, origin="lower", extent=ext, aspect="auto", cmap="RdYlGn_r",
                       vmin=max(0.5, NR.min()), vmax=min(1.4, NR.max()))
    ax[0].plot([g[0], g[-1]], [g[0], g[-1]], "k--", lw=1.6, label=r"global-g line ($g_1=g_2$)")
    o = B["opt"]; bd = B["best_diag"]
    ax[0].scatter([o["g2"]], [o["g1"]], marker="*", s=430, color="gold", edgecolor="black",
                  linewidth=1.4, zorder=6, label=f"two-knob optimum  NRMSE={o['nrmse']:.3f}")
    ax[0].scatter([bd["g"]], [bd["g"]], marker="o", s=150, color="black", zorder=6,
                  label=f"best global g  NRMSE={bd['nrmse']:.3f}")
    ax[0].set_xlabel(r"group-2 phase  $g_2/\pi$"); ax[0].set_ylabel(r"group-1 phase  $g_1/\pi$")
    ax[0].set_title("(a) Task NRMSE over two independent knobs")
    ax[0].legend(fontsize=9, loc="upper right", framealpha=0.92)
    fig.colorbar(im0, ax=ax[0]).set_label("test NRMSE")
    im1 = ax[1].imshow(SO, origin="lower", extent=ext, aspect="auto", cmap="magma")
    ax[1].plot([g[0], g[-1]], [g[0], g[-1]], "w--", lw=1.6)
    ax[1].scatter([o["g2"]], [o["g1"]], marker="*", s=430, color="cyan", edgecolor="black",
                  linewidth=1.4, zorder=6)
    ax[1].set_xlabel(r"group-2 phase  $g_2/\pi$"); ax[1].set_ylabel(r"group-1 phase  $g_1/\pi$")
    ax[1].set_title(r"(b) Operator entanglement $S_{\mathrm{op}}$")
    fig.colorbar(im1, ax=ax[1]).set_label(r"$S_{\mathrm{op}}$")
    rel = 100 * (bd["nrmse"] - o["nrmse"]) / bd["nrmse"]
    fig.suptitle("Extension 2 - A second independent g knob beats the best single global g "
                 f"by {rel:.0f}%  (N=8, Chain)\nThe optimum lies OFF the diagonal: one region "
                 "ordered (memory), one region critical (nonlinearity)", fontsize=13, y=1.02)
    fig.tight_layout(); save(fig, "fig2_twoknob_phase_diagram.png")


def fig_nknobs():
    B = R["B2"]; G = np.array(B["G_list"]); m = np.array(B["mean"]); s = np.array(B["std"])
    fig, ax = plt.subplots(figsize=(8.2, 5.6))
    ax.errorbar(G, m, yerr=s, fmt="-o", color=PURPLE, ms=10, lw=2.4, capsize=5,
                label=f"best multi-g NRMSE (N={B['N']}, {len(B['raw'][G[0]])} seeds)")
    ax.axhline(m[0], color="black", ls="--", lw=1.4,
               label=f"single global g baseline (G=1) = {m[0]:.3f}")
    for gi, mi in zip(G, m):
        ax.annotate(f"{mi:.3f}", (gi, mi), textcoords="offset points", xytext=(9, 9),
                    fontsize=10, color=PURPLE)
    ax.set_xlabel("number of independently tunable g knobs  G")
    ax.set_ylabel("best test NRMSE (k-Pauli, k=3)"); ax.set_xticks(G)
    ax.set_title("Extension 2 - More controllable g parameters -> monotonically lower error\n"
                 "(nested refining partition, warm-started; G=8 is per-edge here)")
    ax.grid(alpha=0.3); ax.legend(fontsize=10)
    rel = 100 * (m[0] - m[-1]) / m[0]
    ax.text(0.97, 0.6, f"G=1 -> G={G[-1]}\n{rel:.0f}% relative\nNRMSE reduction",
            transform=ax.transAxes, ha="right", fontsize=11, color=PURPLE,
            bbox=dict(boxstyle="round", fc="#f4ecf7", ec=PURPLE))
    fig.tight_layout(); save(fig, "fig3_nknobs.png")


def fig_profile():
    B = R["B2"]; prof = np.array(B["profiles"][B["G_list"][-1]])
    topo = chain_topology(B["N"]); coords = {i: topo.coords[i] for i in range(topo.N)}
    fig, ax = plt.subplots(figsize=(10.5, 4.4))
    segs, vals = [], []
    for (i, j), gv in zip(topo.edges, prof):
        (r1, c1), (r2, c2) = coords[i], coords[j]
        segs.append([(c1, -r1), (c2, -r2)]); vals.append(gv)
    lc = LineCollection(segs, cmap="plasma", linewidths=10, array=np.array(vals))
    lc.set_clim(0, 0.5); ax.add_collection(lc)
    for plaq in topo.plaquettes:
        pts = np.array([(coords[q][1], -coords[q][0]) for q in plaq])
        ax.add_patch(mpatches.Polygon(pts, closed=True, facecolor="#fff3cf",
                                      alpha=0.55, edgecolor="none", zorder=0))
    for i, (r, c) in coords.items():
        ax.scatter([c], [-r], s=440, color="#222", zorder=5)
        ax.text(c, -r, str(i), color="white", ha="center", va="center",
                fontsize=10, fontweight="bold", zorder=6)
    ax.set_aspect("equal"); ax.axis("off"); ax.autoscale()
    fig.colorbar(lc, ax=ax, fraction=0.05, pad=0.02).set_label(r"optimised per-edge phase  $g_e/\pi$")
    ax.set_title(f"Extension 2 - Optimised heterogeneous g-profile (N={B['N']} Chain)\n"
                 f"global-g NRMSE {B['mean'][0]:.3f}  ->  per-edge NRMSE {B['mean'][-1]:.3f}; "
                 "dark = ordered/memory, bright = critical/scrambling")
    fig.tight_layout(); save(fig, "fig4_g_profile.png")


def fig_combined():
    C = R["C"]; Ns = np.array(C["N_list"])
    gm, gs = np.array(C["global_mean"]), np.array(C["global_std"])
    mm, ms = np.array(C["multiG_mean"]), np.array(C["multiG_std"])
    gain = gm - mm
    fig, ax = plt.subplots(1, 2, figsize=(13, 5.3))
    ax[0].errorbar(Ns, gm, yerr=gs, fmt="-o", color=BLUE, ms=8, lw=2.2, capsize=4,
                   label="best single global g")
    ax[0].errorbar(Ns, mm, yerr=ms, fmt="-s", color=PURPLE, ms=8, lw=2.2, capsize=4,
                   label=f"best multi-g (G={C['G']})")
    ax[0].fill_between(Ns, mm, gm, color="gold", alpha=0.25)
    ax[0].set_xlabel("number of qubits  N"); ax[0].set_ylabel("best test NRMSE")
    ax[0].set_xticks(Ns); ax[0].grid(alpha=0.3); ax[0].legend(fontsize=10)
    ax[0].set_title("(a) Global-g vs multi-g across reservoir sizes")
    ax[1].bar(Ns, gain, width=0.9, color=GREEN, alpha=0.85, edgecolor="black")
    for n, gv in zip(Ns, gain):
        ax[1].annotate(f"{gv:+.2f}", (n, gv), textcoords="offset points",
                       xytext=(0, 4), ha="center", fontsize=10)
    ax[1].set_xlabel("number of qubits  N")
    ax[1].set_ylabel(r"NRMSE gain (global $-$ multi-g)")
    ax[1].set_xticks(Ns); ax[1].grid(alpha=0.3, axis="y")
    ax[1].set_title("(b) Multi-g advantage grows with reservoir size")
    fig.suptitle("Combined - the two extensions reinforce each other: more qubits give more "
                 "edges to tune,\nso the multi-g advantage widens with N", fontsize=13, y=1.02)
    fig.tight_layout(); save(fig, "fig5_combined_advantage.png")

In [ ]:
fig_scaling(); fig_twoknob(); fig_nknobs(); fig_profile(); fig_combined()